# Permanente evaluatie - vaardigheidstoets 2 (OEFENTEST)
Steel Plates Faults - 2ITAI - AP Hogeschool

## Opgave

De dataset `Faults.NNA` bevat metingen van defecten op stalen platen uit een productielijn. Elke rij is één defect. De kolomnamen staan in het bestand `Faults27x7_var` (één naam per lijn, het bestand `Faults.NNA` heeft **geen header** en is gescheiden door tabs/spaties).

De laatste 7 kolommen (`Pastry`, `Z_Scratch`, `K_Scatch`, `Stains`, `Dirtiness`, `Bumps`, `Other_Faults`) geven het type defect aan als **one-hot encoding**: per rij staat er precies één `1`. Jullie moeten proberen om dit type defect als label te reproduceren via clusters. Zet deze 7 kolommen dus eerst om naar **één labelkolom** `fault_type`, haal ze van je dataset en hou het label bij om op het einde met jouw gemaakte clusters te vergelijken.

Pas op deze dataset een datareductietechniek toe, en cluster vervolgens de gereduceerde dataset zo goed mogelijk. Zorg voor preprocessing waar nodig, en zoek naar een optimaal model. Gebruik **twee verschillende clusteringalgoritmes** en kies het beste. **Indien je bepaalde keuzes maakt, schrijf kort in de notebook waarom**. Vergelijk op het einde of de labels van je clusteringalgoritme overeenkomen met de originele labels `fault_type`, met behulp van een confusion matrix waarbij je de labels meegeeft en de clusters (clusters zijn het resultaat van `fit_predict()`). Trek hier conclusies uit en noteer deze in de notebook.

### Inleveren:
- Hernoem je bestand naar **PE2-VoornaamNaam.ipynb** waarbij je je *eigen naam* invult.

Jullie krijgen **60 minuten** de tijd.

### Kolommen dataset:

- X_Minimum, X_Maximum, Y_Minimum, Y_Maximum : positie van het defect op de plaat
- Pixels_Areas : oppervlakte van het defect in pixels
- X_Perimeter, Y_Perimeter : omtrek in x- en y-richting
- Sum_of_Luminosity, Minimum_of_Luminosity, Maximum_of_Luminosity : helderheid van het defect
- Length_of_Conveyer : lengte van de transportband
- TypeOfSteel_A300, TypeOfSteel_A400 : soort staal (0/1)
- Steel_Plate_Thickness : dikte van de plaat
- Edges_Index, Empty_Index, Square_Index, Outside_X_Index, Edges_X_Index, Edges_Y_Index, Outside_Global_Index : vormindexen
- LogOfAreas, Log_X_Index, Log_Y_Index : logaritmische versies van oppervlakte en afmetingen
- Orientation_Index, Luminosity_Index, SigmoidOfAreas : afgeleide kenmerken
- Pastry, Z_Scratch, K_Scatch, Stains, Dirtiness, Bumps, Other_Faults : type defect (one-hot) → dit is het label dat we via clustering proberen te reproduceren.

### Indicatieve puntenverdeling /20
- goede algoritmes: 2 keer 4 ptn
- goede beschrijving over keuze van deze algoritmes: 4 ptn
- overige noodzakelijke stappen: 8 ptn
- Hierna wordt er gekeken naar code stijl

### Tips (enkel lezen als je vastzit)
<details>
<summary>Klik om te tonen</summary>

- Inlezen: `pd.read_csv(pad, sep=r"\s+", header=None, names=kolomnamen)`
- One-hot naar één label: `df[fault_cols].idxmax(axis=1)`
- Zijn alle kolommen nuttig? Kijk eens naar `TypeOfSteel_A300` en `TypeOfSteel_A400` samen.
- Hoeveel types defect zijn er? Wat betekent dat voor je keuze van `k`?
- De klassen zijn niet even groot: kijk naar `value_counts()` voor je conclusies trekt.
</details>


## Oplossing

In [7]:
# nodige imports
import numpy as np
import pandas as pd

In [23]:
# data inladen
data_path = "./steel_plates/Faults.NNA"
names_path = "./steel_plates/Faults27x7_var"


In [25]:
# kolomnamen inladen
with open(names_path, "r") as f:
    colnames = [line.strip() for line in f.readlines()]

# Data laden
df = pd.read_csv(data_path, sep="\\s", header=None, names=colnames)

df.head()

/var/folders/nq/dkdpj8m968v88lf2j6w38bpr0000gn/T/ipykernel_5449/3460994621.py:6: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  df = pd.read_csv(data_path, sep="\\s", header=None, names=colnames)


,X_Minimum,X_Maximum,Y_Minimum,Y_Maximum,Pixels_Areas,X_Perimeter,Y_Perimeter,Sum_of_Luminosity,Minimum_of_Luminosity,Maximum_of_Luminosity,...,Orientation_Index,Luminosity_Index,SigmoidOfAreas,Pastry,Z_Scratch,K_Scatch,Stains,Dirtiness,Bumps,Other_Faults
0,42,50,270900,270944,267,17,44,24220,76,108,...,0.8182,-0.2913,0.5822,1,0,0,0,0,0,0
1,645,651,2538079,2538108,108,10,30,11397,84,123,...,0.7931,-0.1756,0.2984,1,0,0,0,0,0,0
2,829,835,1553913,1553931,71,8,19,7972,99,125,...,0.6667,-0.1228,0.2150,1,0,0,0,0,0,0
3,853,860,369370,369415,176,13,45,18996,99,126,...,0.8444,-0.1568,0.5212,1,0,0,0,0,0,0
4,1289,1306,498078,498335,2409,60,260,246930,37,126,...,0.9338,-0.1992,1.0000,1,0,0,0,0,0,0


In [ ]:
df.shape #om te zien hvl kolommen ik heb 


(1941, 34)

In [31]:
34-7

27

In [36]:
X = df.iloc[:, :27]
y = df.iloc[:, 27:]

# Ontbrekende waarden invullen met het kolomgemiddelde
X = X.fillna(X.mean())

In [37]:
#Met zowel Kmeans als dbscan werken we met afstanden dus standisere is belangrijk en meestal staan data in heel verschillende schalen. Pca is gebasseerd op variatie en zoekt dominante richtingen. als we niet schalen zullen features grote waardes krijgen
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [41]:
# Na dan we data reductie toepassen
from sklearn.decomposition import PCA

pca = PCA(n_components=0.95)
X_pca_steel = pca.fit_transform(X_scaled)

n_components_steel = X_pca_steel.shape[1]
reduction_pca_steel = (1-n_components_steel/X.shape[1]) * 100

print(f"Aantal componenten Voor reductie: {X.shape[1]}")
print(f"Aantal componenten na reductie: {n_components_steel}")
print(f"Gereduceerde data: {reduction_pca_steel:.2f}")

Aantal componenten Voor reductie: 27
Aantal componenten na reductie: 13
Gereduceerde data: 51.85


In [ ]:
y.idxmax(axis=1).unique().shape #om te zien hbl labels ik heb.

(7,)

In [53]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score


kmeans = KMeans(n_clusters=7, random_state=42, n_init=10).fit(X_scaled)
sil = silhouette_score(X_pca_steel, kmeans.labels_)
print(sil)

0.20657500009761484


In [55]:
from sklearn.cluster import DBSCAN


dbscan_params = [(0.5, 5), (0.5, 10),
                 (1.0, 5), (1.0, 10),
                 (1.5, 5), (1.5, 10),
                 (2.0, 5), (2.0, 10),
                 (3.0, 5), (3.0, 10)]

best_sil = -1
best_params = None

for eps, min_samples in dbscan_params:
    dbscan = DBSCAN(eps=eps, min_samples=min_samples).fit(X_pca_steel)
    
    labels = dbscan.labels_
    
    if len(set(labels)) >=2:
        sil = silhouette_score(X_pca_steel, labels)
        
        if sil > best_sil:
            best_sil = sil
            best_params = (eps, min_samples)
            
print(best_params, best_sil)
    


(0.5, 10) 0.10834363700419825
